# Predict New York City Airbnb Prices
**AIML Minor Project 2 — Linear Regression**
**Name:** Jaisurya Ramesh

This notebook follows the seven steps in the assignment using the AB_NYC_2019 dataset. Run the cells from top to bottom.

## Step 1: Exploratory Data Analysis and Data Auditing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
data_url = 'https://huggingface.co/datasets/gradio/NYC-Airbnb-Open-Data/resolve/main/AB_NYC_2019.csv'
df = pd.read_csv(data_url)
print('Dataset shape:', df.shape)
display(df.head())
display(df.dtypes)
display(df.describe(include='all').T)


In [ ]:
numerical_columns = df.select_dtypes(include=np.number).columns.tolist()
categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
discrete_columns = ['id', 'host_id', 'minimum_nights', 'number_of_reviews', 'calculated_host_listings_count', 'availability_365']
continuous_columns = ['latitude', 'longitude', 'price', 'reviews_per_month']
print('Numerical:', numerical_columns)
print('Categorical:', categorical_columns)
print('Discrete:', [c for c in discrete_columns if c in df.columns])
print('Continuous:', [c for c in continuous_columns if c in df.columns])


In [ ]:
missing_summary = pd.DataFrame({'Missing Values': df.isnull().sum(), 'Missing Percentage': (df.isnull().mean() * 100).round(2)})
display(missing_summary.sort_values('Missing Values', ascending=False))


## Step 2: Data Cleaning and Handling Missing Values

In [ ]:
clean = df.copy()
clean = clean.drop(columns=['last_review'])
zero_review_rows = clean['number_of_reviews'].eq(0)
clean.loc[zero_review_rows & clean['reviews_per_month'].isna(), 'reviews_per_month'] = 0
clean['reviews_per_month'] = clean['reviews_per_month'].fillna(clean['reviews_per_month'].median())
drop_columns = ['id', 'name', 'host_id', 'host_name', 'neighbourhood']
clean = clean.drop(columns=[c for c in drop_columns if c in clean.columns])
print('Shape after cleaning:', clean.shape)
print('Missing values remaining:', clean.isnull().sum().sum())


## Step 3: Outlier Identification and Treatment

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=df['price'], ax=axes[0]); axes[0].set_title('Price before treatment')
sns.boxplot(x=df['minimum_nights'], ax=axes[1]); axes[1].set_title('Minimum nights before treatment')
plt.tight_layout(); plt.show()
price_95 = clean['price'].quantile(0.95)
nights_95 = clean['minimum_nights'].quantile(0.95)
rows_before = len(clean)
clean = clean[(clean['price'] > 0) & (clean['price'] <= price_95) & (clean['minimum_nights'] < 365) & (clean['minimum_nights'] <= nights_95)].copy()
print('Price 95th percentile:', price_95)
print('Minimum nights 95th percentile:', nights_95)
print('Rows before:', rows_before, 'Rows after:', len(clean))
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=clean['price'], ax=axes[0]); axes[0].set_title('Price after treatment')
sns.boxplot(x=clean['minimum_nights'], ax=axes[1]); axes[1].set_title('Minimum nights after treatment')
plt.tight_layout(); plt.show()


## Step 4: Feature Selection and Categorical Encoding

In [ ]:
print('Neighbourhood count:', df['neighbourhood'].nunique())
print('Neighbourhood group count:', df['neighbourhood_group'].nunique())
clean = clean.drop(columns=['neighbourhood'], errors='ignore')
clean = pd.get_dummies(clean, columns=['neighbourhood_group', 'room_type'], drop_first=True, dtype=int)
display(clean.head())
print('Encoded shape:', clean.shape)


## Step 5: Feature-Target Separation and Validation Split

In [ ]:
X = clean.drop(columns=['price'])
y = clean['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print('Training:', X_train.shape, y_train.shape)
print('Testing:', X_test.shape, y_test.shape)


## Step 6: Model Training and Evaluation

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
print(f'R-squared: {r2:.4f}')
print(f'MAE: ${mae:,.2f}')
print(f'MSE: {mse:,.2f}')
print(f'RMSE: ${rmse:,.2f}')


## Step 7: Residual Analysis and Interpretation

In [ ]:
results = pd.DataFrame({'Actual Price': y_test, 'Predicted Price': y_pred})
results['Residual'] = results['Actual Price'] - results['Predicted Price']
plt.figure(figsize=(7, 6))
sns.scatterplot(data=results, x='Actual Price', y='Predicted Price', alpha=0.35)
low = min(results['Actual Price'].min(), results['Predicted Price'].min())
high = max(results['Actual Price'].max(), results['Predicted Price'].max())
plt.plot([low, high], [low, high], linestyle='--', color='red')
plt.title('Actual Prices vs Predicted Prices'); plt.tight_layout(); plt.show()
plt.figure(figsize=(7, 5))
sns.scatterplot(data=results, x='Predicted Price', y='Residual', alpha=0.35)
plt.axhline(0, linestyle='--', color='red'); plt.title('Residual Plot'); plt.tight_layout(); plt.show()
plt.figure(figsize=(7, 5)); sns.histplot(results['Residual'], bins=40, kde=True)
plt.title('Distribution of Prediction Errors'); plt.tight_layout(); plt.show()
results['Price Group'] = np.where(results['Actual Price'] <= results['Actual Price'].median(), 'Lower-priced listings', 'Higher-priced listings')
group_errors = results.groupby('Price Group').apply(lambda g: pd.Series({'Listings': len(g), 'MAE ($)': mean_absolute_error(g['Actual Price'], g['Predicted Price'])}), include_groups=False)
display(group_errors)
better_group = group_errors['MAE ($)'].idxmin()
print('Lower average dollar error:', better_group)
